# Notebook 8 - Batter Season Stats

**Goal**: Pre-compute all batter statistics used by the Batters dashboard page so the page callback only needs to filter and render, not compute.

**Why pre-compute?** The batters page currently runs groupby/aggregation on 279k delivery rows every time the season filter changes. Moving that computation here means the app is faster and the analysis logic is in one place.

**Output**: `batter_phase_season.csv` - one row per batter per season with stats for each phase (powerplay, middle, death) plus RAPA.

Columns:
- `season, batter` - identifiers
- `pp_balls, pp_runs, pp_sr, pp_boundary_pct, pp_dot_pct` - powerplay stats
- `mid_balls, mid_runs, mid_sr, mid_boundary_pct, mid_dot_pct` - middle overs
- `death_balls, death_runs, death_sr, death_boundary_pct, death_dot_pct` - death overs
- `total_balls, total_runs` - overall volume
- `avg_batting_position` - mean position across all innings that season (used for death specialist filter: avg > 5)
- `rapa` - Runs Above Phase Average: total runs minus what a league-average batter would score facing the same balls in the same phases

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

## 2. Load Data

I load all seasons (no era filter) so the CSV covers the full history. The dashboard callback filters by the selected season window.

In [2]:
df = pd.read_csv('../data/processed/deliveries.csv')
df = df[(df['super_over'] == False) & (df['is_wide'] == False)].copy()

print(f'Rows (no super overs, no wides): {len(df):,}')
print(f'Seasons: {df["season"].min()} - {df["season"].max()}')
print(f'Unique batters: {df["batter"].nunique():,}')

Rows (no super overs, no wides): 277,831
Seasons: 2008 - 2026
Unique batters: 726


## 3. Phase Stats Per Batter Per Season

I compute balls faced, runs scored, strike rate, boundary %, and dot ball % for each of the three phases. These are the raw stats used by the powerplay specialists, middle anchors, and death specialists leaderboards.

In [3]:
# Group by season, batter, and phase to get per-phase counts
phase_stats = (
    df.groupby(['season', 'batter', 'phase'])
    .agg(
        balls       = ('batter_runs', 'count'),
        runs        = ('batter_runs', 'sum'),
        boundaries  = ('is_boundary_4', 'sum'),  # 4s
        sixes       = ('is_boundary_6', 'sum'),  # 6s
        dots        = ('is_dot',        'sum'),
    )
    .reset_index()
)

phase_stats['boundary_pct'] = (phase_stats['boundaries'] + phase_stats['sixes']) / phase_stats['balls'] * 100
phase_stats['dot_pct']      = phase_stats['dots'] / phase_stats['balls'] * 100
phase_stats['sr']           = phase_stats['runs'] / phase_stats['balls'] * 100

print(f'Phase-batter-season rows: {len(phase_stats):,}')
print(phase_stats.head(6))

Phase-batter-season rows: 6,080
   season    batter      phase  balls  runs  boundaries  sixes  dots  \
0    2008  A Chopra      death      2     1           0      0     1   
1    2008  A Chopra     middle     15    18           2      0     5   
2    2008  A Chopra  powerplay     35    23           3      0    21   
3    2008  A Kumble      death     17    13           1      0     7   
4    2008  A Mishra      death     11     6           0      0     6   
5    2008  A Mishra     middle     31    31           3      0    10   

   boundary_pct  dot_pct      sr  
0         0.000   50.000  50.000  
1        13.333   33.333 120.000  
2         8.571   60.000  65.714  
3         5.882   41.176  76.471  
4         0.000   54.545  54.545  
5         9.677   32.258 100.000  


In [4]:
# Pivot from long format (one row per phase) to wide (one row per batter-season)
# Each metric gets a pp_, mid_, death_ prefix
phase_map = {'powerplay': 'pp', 'middle': 'mid', 'death': 'death'}

wide = {}
for phase_name, prefix in phase_map.items():
    p = phase_stats[phase_stats['phase'] == phase_name].copy()
    p = p.rename(columns={
        'balls':        f'{prefix}_balls',
        'runs':         f'{prefix}_runs',
        'sr':           f'{prefix}_sr',
        'boundary_pct': f'{prefix}_boundary_pct',
        'dot_pct':      f'{prefix}_dot_pct',
    })
    wide[phase_name] = p[['season', 'batter', f'{prefix}_balls', f'{prefix}_runs',
                           f'{prefix}_sr', f'{prefix}_boundary_pct', f'{prefix}_dot_pct']]

# Start from powerplay, then merge in middle and death
batter_wide = wide['powerplay'].merge(wide['middle'],  on=['season', 'batter'], how='outer')
batter_wide = batter_wide.merge(wide['death'], on=['season', 'batter'], how='outer')

# Fill NaN (batter didn't face any balls in that phase that season)
phase_cols = [c for c in batter_wide.columns if c not in ['season', 'batter']]
batter_wide[phase_cols] = batter_wide[phase_cols].fillna(0)

print(f'Wide format rows (batter-season): {len(batter_wide):,}')
print(batter_wide.head(3))

Wide format rows (batter-season): 2,924
   season    batter  pp_balls  pp_runs  pp_sr  pp_boundary_pct  pp_dot_pct  \
0    2008  A Chopra    35.000   23.000 65.714            8.571      60.000   
1    2008  A Kumble     0.000    0.000  0.000            0.000       0.000   
2    2008  A Mishra     0.000    0.000  0.000            0.000       0.000   

   mid_balls  mid_runs  mid_sr  mid_boundary_pct  mid_dot_pct  death_balls  \
0     15.000    18.000 120.000            13.333       33.333        2.000   
1      0.000     0.000   0.000             0.000        0.000       17.000   
2     31.000    31.000 100.000             9.677       32.258       11.000   

   death_runs  death_sr  death_boundary_pct  death_dot_pct  
0       1.000    50.000               0.000         50.000  
1      13.000    76.471               5.882         41.176  
2       6.000    54.545               0.000         54.545  


## 4. Overall Volume and Batting Position

Total balls and runs across all phases, plus average batting position. The batting position average is needed by the death specialist filter: a death specialist should have avg position > 5, which excludes openers who occasionally survive into the death overs.

In [5]:
# Total balls and runs per batter per season (across all phases)
overall = (
    df.groupby(['season', 'batter'])
    .agg(
        total_balls = ('batter_runs', 'count'),
        total_runs  = ('batter_runs', 'sum'),
    )
    .reset_index()
)

# Average batting position per batter per season
# batting_position is the position in the batting order for that innings
avg_position = (
    df.groupby(['season', 'batter'])['batting_position']
    .mean()
    .reset_index(name='avg_batting_position')
)

batter_wide = batter_wide.merge(overall,       on=['season', 'batter'])
batter_wide = batter_wide.merge(avg_position,  on=['season', 'batter'])

print('Overall + position merged. Sample:')
print(batter_wide[['season', 'batter', 'total_balls', 'total_runs', 'avg_batting_position']].head(5))

Overall + position merged. Sample:
   season    batter  total_balls  total_runs  avg_batting_position
0    2008  A Chopra           52          42                 2.635
1    2008  A Kumble           17          13                10.000
2    2008  A Mishra           42          37                 7.119
3    2008  A Mukund            1           0                 8.000
4    2008   A Nehra           13           3                10.538


## 5. RAPA - Runs Above Phase Average

Runs Above Phase Average answers: did this batter score more or fewer runs than a league-average batter would have scored facing the same number of balls in the same phases?

For each ball faced:
```
contribution = batter_runs - league_avg_sr_for_that_phase / 100
```
Summing this gives RAPA. A positive RAPA means the batter outperformed the league phase average.

**I compute league averages per season** - so a 2021 batter is judged against 2021 averages, not the 2008-2026 average. This keeps the comparison era-appropriate.

In [6]:
# League SR per phase per season (runs per ball * 100, across all qualified deliveries)
league_phase_sr = (
    df.groupby(['season', 'phase'])
    .agg(total_runs=('batter_runs', 'sum'), total_balls=('batter_runs', 'count'))
    .reset_index()
)
league_phase_sr['league_sr'] = league_phase_sr['total_runs'] / league_phase_sr['total_balls'] * 100

print('League SR per phase per season (2021-2026 sample):')
print(league_phase_sr[league_phase_sr['season'] >= 2021].to_string(index=False))

League SR per phase per season (2021-2026 sample):
 season     phase  total_runs  total_balls  league_sr
   2021     death        4664         3189    146.253
   2021    middle        7751         6424    120.657
   2021 powerplay        5301         4339    122.171
   2022     death        6401         3925    163.083
   2022    middle       10153         7937    127.920
   2022 powerplay        6498         5348    121.503
   2023     death        6473         3992    162.149
   2023    middle       10663         7924    134.566
   2023 powerplay        7292         5321    137.042
   2024     death        6356         3658    173.756
   2024    middle       10668         7580    140.739
   2024 powerplay        7633         5136    148.618
   2025     death        6185         3604    171.615
   2025    middle       11146         7721    144.360
   2025 powerplay        7955         5251    151.495
   2026     death        3166         1926    164.382
   2026    middle        6091  

In [7]:
# RAPA per batter per season = sum over all phases of: batter_runs - (league_sr/100 * balls_faced)
# For each phase: RAPA_contribution = phase_runs - (league_sr/100 * phase_balls)

# Look up the league SR for each phase
for phase_name, prefix in [('powerplay', 'pp'), ('middle', 'mid'), ('death', 'death')]:
    league = league_phase_sr[league_phase_sr['phase'] == phase_name][['season', 'league_sr']]
    league = league.rename(columns={'league_sr': f'{prefix}_league_sr'})
    batter_wide = batter_wide.merge(league, on='season', how='left')

# RAPA = sum of (actual_runs - expected_runs) across all three phases
batter_wide['rapa'] = (
    (batter_wide['pp_runs']    - batter_wide['pp_league_sr']    / 100 * batter_wide['pp_balls']) +
    (batter_wide['mid_runs']   - batter_wide['mid_league_sr']   / 100 * batter_wide['mid_balls']) +
    (batter_wide['death_runs'] - batter_wide['death_league_sr'] / 100 * batter_wide['death_balls'])
)

# Drop the intermediate league_sr columns - they're only needed for the calculation
batter_wide = batter_wide.drop(columns=['pp_league_sr', 'mid_league_sr', 'death_league_sr'])

print('RAPA distribution:')
print(batter_wide[batter_wide['total_balls'] >= 50]['rapa'].describe().round(2))

RAPA distribution:
count   1395.000
mean       6.340
std       36.740
min     -120.030
25%      -16.960
50%       -1.730
75%       21.520
max      224.950
Name: rapa, dtype: float64


## 6. Preview and Save

In [8]:
# Preview: top 10 RAPA in 2024 (min 50 balls)
print('Top 10 RAPA leaders in 2024 (min 50 balls):')
preview = (
    batter_wide[(batter_wide['season'] == 2024) & (batter_wide['total_balls'] >= 50)]
    .sort_values('rapa', ascending=False)
    .head(10)
)[['batter', 'total_balls', 'total_runs', 'rapa', 'avg_batting_position']]
print(preview.to_string(index=False))

print('\nTop 10 death SR in 2024 (min 50 death balls, avg position > 5):')
print(
    batter_wide[
        (batter_wide['season'] == 2024) &
        (batter_wide['death_balls'] >= 50) &
        (batter_wide['avg_batting_position'] > 5)
    ]
    .sort_values('death_sr', ascending=False)
    .head(10)[['batter', 'death_balls', 'death_sr', 'avg_batting_position']]
    .to_string(index=False)
)

Top 10 RAPA leaders in 2024 (min 50 balls):
         batter  total_balls  total_runs    rapa  avg_batting_position
Abhishek Sharma          237         484 137.134                 2.030
        TM Head          296         567 135.207                 1.385
J Fraser-McGurk          141         330 123.207                 1.901
      SP Narine          270         488  91.043                 2.078
        PD Salt          239         435  85.038                 1.000
     RM Patidar          223         395  74.118                 3.807
       T Stubbs          198         378  67.089                 5.162
       N Pooran          280         499  62.463                 5.175
      H Klaasen          280         479  54.255                 4.993
       SA Yadav          206         345  45.383                 3.379

Top 10 death SR in 2024 (min 50 death balls, avg position > 5):
         batter  death_balls  death_sr  avg_batting_position
       T Stubbs       96.000   262.500           

In [9]:
# Round floats to keep file size manageable
float_cols = [c for c in batter_wide.columns if batter_wide[c].dtype == float]
batter_wide[float_cols] = batter_wide[float_cols].round(4)

batter_wide.to_csv('../data/processed/batter_phase_season.csv', index=False)
print(f'Saved {len(batter_wide):,} rows to data/processed/batter_phase_season.csv')
print(f'Columns: {list(batter_wide.columns)}')

Saved 2,924 rows to data/processed/batter_phase_season.csv
Columns: ['season', 'batter', 'pp_balls', 'pp_runs', 'pp_sr', 'pp_boundary_pct', 'pp_dot_pct', 'mid_balls', 'mid_runs', 'mid_sr', 'mid_boundary_pct', 'mid_dot_pct', 'death_balls', 'death_runs', 'death_sr', 'death_boundary_pct', 'death_dot_pct', 'total_balls', 'total_runs', 'avg_batting_position', 'rapa']


## Summary

Pre-computed batter stats for every batter-season in IPL history (2008-2026).

**How the dashboard uses this file**:
- Powerplay specialists: filter `pp_balls >= 50`, sort by `pp_sr`
- Middle anchors: filter `mid_balls >= 50`, sort by `mid_sr`
- Death specialists: filter `death_balls >= 50` AND `avg_batting_position > 5`, sort by `death_sr`
- Season runs leader: filter `total_balls >= 50`, find max `total_runs` per season
- Season efficiency leader (RAPA): filter `total_balls >= 50`, find max `rapa` per season
- Complete batsmen scatter: merge pp and death qualified subsets, plot pp_sr vs death_sr

**Saved output**: `data/processed/batter_phase_season.csv`